# 06 - Application, Testing, Reports, Export

**Purpose:** Run pytest, generate docs from executed artifacts, launch API + dashboard inside Colab, export the portable bundle, optional GitHub push.

**Inputs:** All previous artifacts

**Outputs / persisted artifacts:** docs/*.md (auto-generated), exports/creditiq_export_*.zip, manifest.json with SHA-256 checksums

**Dependencies:** Notebooks 01-05

**Execution order:** 6 of 6

**Validation checks:** pytest passes (executed), API /health ok, dashboard loads, export manifest verifies

_All logic lives in `src/creditiq`; cells are safe to re-run (completed stages are skipped, artifacts live on Google Drive)._

In [ ]:
# --- Colab setup (safe to re-run) ---
import os, sys, subprocess
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ROOT = Path("/content/drive/MyDrive/CreditIQ") if Path("/content/drive/MyDrive").exists() else Path.cwd().parent
os.environ["CREDITIQ_ROOT"] = str(ROOT); sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "src").exists():
    raise SystemExit(f"Project not found at {ROOT}. Unzip CreditIQ into MyDrive/CreditIQ first.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=False)
from creditiq.config import get_paths, load_config, read_state
paths = get_paths(); cfg = load_config(paths)
print("Project root:", ROOT); print("Last completed stage:", read_state(paths).get("last_completed"))

In [ ]:
!cd "$CREDITIQ_ROOT" && python -m pytest -q 2>&1 | tail -15

In [ ]:
from creditiq import pipeline
pipeline.run("report"); pipeline.run("export")

**Launch API (port 8000) and dashboard (port 8501).** Colab proxies the ports privately to your session - no public tunnel, no secrets exposed. Colab is not permanent hosting.

In [ ]:
import subprocess, time
api = subprocess.Popen([sys.executable, "-m", "uvicorn", "api.main:app", "--port", "8000"], cwd=ROOT, env={**os.environ})
dash = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "dashboard/app.py", "--server.port", "8501", "--server.headless", "true",
                         "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"], cwd=ROOT, env={**os.environ})
time.sleep(15)
import requests; print(requests.get("http://localhost:8000/health").json())

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8501)      # dashboard (opens a Colab-proxied link)
# output.serve_kernel_port_as_iframe(8501, height=800)   # or inline

In [ ]:
# Optional: push SOURCE to GitHub (token from Colab Secrets, never hardcoded)
# from google.colab import userdata
# token = userdata.get("GITHUB_TOKEN"); repo = "https://{}@github.com/<user>/CreditIQ.git".format(token)
# !cd "$CREDITIQ_ROOT" && git init -q && git add -A && git commit -qm "CreditIQ" && git branch -M main && git remote add origin "{repo}" && git push -u origin main

In [ ]:
api.terminate(); dash.terminate()   # stop servers when done